In [ ]:
# =============================================================================
# Setup & Chain-of-Thought vs Direct Answer
# =============================================================================
# Chain-of-thought (CoT) prompting asks the model to reason step-by-step
# before giving a final answer. Like requiring an adjuster to show their
# work — not just "denied" but WHY denied.
# =============================================================================

import boto3
import json
from datetime import datetime

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')
model_id = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

def ask(prompt, system=None, max_tokens=2048, temperature=0.0):
    kwargs = {
        'modelId': model_id,
        'messages': [{'role': 'user', 'content': [{'text': prompt}]}],
        'inferenceConfig': {'maxTokens': max_tokens, 'temperature': temperature}
    }
    if system:
        kwargs['system'] = [{'text': system}]
    response = bedrock.converse(**kwargs)
    return {
        'text': response['output']['message']['content'][0]['text'],
        'input_tokens': response['usage']['inputTokens'],
        'output_tokens': response['usage']['outputTokens'],
        'stop_reason': response['stopReason']
    }

def parse_llm_json(text):
    """Strip markdown fencing and parse JSON from LLM output."""
    raw = text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

# --- A coverage determination that requires reasoning ---
coverage_scenario = """
COVERAGE DETERMINATION REQUEST

Policy: HO-3 Special Form (Open Perils on Dwelling, Named Perils on Personal Property)
Policyholder: Robert and Diana Whitfield
Dwelling Coverage: $520,000 | Personal Property: $260,000 | Deductible: $2,500

Incident: On February 15, 2026, the Whitfields returned from a 3-week vacation 
to find extensive water damage throughout their home. Investigation revealed:

1. A slow leak developed in the upstairs bathroom supply line
2. Water ran continuously for an estimated 10-14 days
3. Damage includes: buckled hardwood floors (main level), saturated drywall 
   (multiple rooms), mold growth behind walls, damaged personal electronics 
   and furniture
4. The home's water shutoff valve was not turned off before the trip
5. A neighbor had agreed to "check on the house" but only drove by weekly
6. The plumber found the supply line fitting was original to the 2004 construction
   and showed signs of gradual corrosion

Question: What is covered, what may be excluded, and what's the recommended action?
"""

# --- Approach 1: Direct answer (no CoT) ---
print("=" * 70)
print("APPROACH 1: Direct Answer (No Chain-of-Thought)")
print("=" * 70)

direct_prompt = f"""You are a claims adjuster. Provide a coverage determination 
for the following claim.

{coverage_scenario}"""

result_direct = ask(direct_prompt)
print(result_direct['text'])
print(f"\n[Tokens: {result_direct['input_tokens']} in / {result_direct['output_tokens']} out]")

# --- Approach 2: Chain-of-thought ---
print("\n" + "=" * 70)
print("APPROACH 2: Chain-of-Thought Reasoning")
print("=" * 70)

cot_prompt = f"""You are a claims adjuster. Provide a coverage determination 
for the following claim.

Think through this step-by-step:
1. First, identify the cause of loss and classify the peril
2. Then, check if this peril is covered under the HO-3 form
3. Next, consider any applicable exclusions or limitations
4. Evaluate whether the policyholder's actions affect coverage
5. Assess each category of damage separately
6. Finally, provide your coverage determination with reasoning

{coverage_scenario}"""

result_cot = ask(cot_prompt)
print(result_cot['text'])
print(f"\n[Tokens: {result_cot['input_tokens']} in / {result_cot['output_tokens']} out]")

# --- Comparison ---
print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)
print(f"{'Approach':<25} {'Input':>10} {'Output':>10} {'Total':>10}")
print("-" * 55)
print(f"{'Direct':<25} {result_direct['input_tokens']:>10} {result_direct['output_tokens']:>10} {result_direct['input_tokens']+result_direct['output_tokens']:>10}")
print(f"{'Chain-of-Thought':<25} {result_cot['input_tokens']:>10} {result_cot['output_tokens']:>10} {result_cot['input_tokens']+result_cot['output_tokens']:>10}")

In [ ]:
# =============================================================================
# Structured CoT — Separating Reasoning from Conclusion
# =============================================================================
# In production, you need the reasoning for auditors but the conclusion for
# your systems. This technique puts them in separate, parseable sections.
# =============================================================================

# --- Technique 1: XML-tagged reasoning separation ---
print("=" * 70)
print("TECHNIQUE 1: XML-Tagged Reasoning Separation")
print("=" * 70)

liability_scenario = """
CLAIM: AUTO-2026-11847

Accident Details:
- Date: February 18, 2026, 7:45 AM, light rain
- Location: Intersection of Oak Street and 5th Avenue, Portland, OR
- Our insured: Karen Walsh, driving a 2024 Subaru Outback northbound on Oak
- Other party: Tyler Briggs, driving a 2022 Ford F-150 eastbound on 5th

Karen states she had a green light and was proceeding through the intersection 
at approximately 30 mph when Briggs ran the red light and struck her passenger 
side. Briggs states the light was yellow turning red and that Walsh was speeding.

Evidence:
- Police report assigns fault to Briggs (cited for running red light)
- Traffic camera footage shows Walsh's light was green for 4 seconds before entry
- Briggs's dashcam shows his light was red before he entered the intersection
- Walsh's speed estimated at 32 mph in a 30 mph zone
- Briggs admitted to the officer he was "running late for work"
- Damage: Walsh vehicle $14,200, Briggs vehicle $8,900
- Walsh: whiplash diagnosis, 3 physical therapy sessions so far
- One passenger in Walsh's vehicle (her daughter, age 14) — no reported injuries
"""

xml_cot_prompt = f"""Analyze this auto liability claim. Structure your response 
in these exact sections:

<reasoning>
Walk through your analysis step-by-step:
- Evaluate each piece of evidence
- Consider comparative negligence
- Assess credibility of each party
- Calculate liability split
</reasoning>

<determination>
State your liability determination clearly.
</determination>

<action_items>
List next steps as bullet points.
</action_items>

CLAIM:
{liability_scenario}"""

result = ask(xml_cot_prompt)
print(result['text'])
print(f"\n[Tokens: {result['input_tokens']} in / {result['output_tokens']} out]")

# --- Parse out each section ---
import re

text = result['text']
reasoning = re.search(r'<reasoning>(.*?)</reasoning>', text, re.DOTALL)
determination = re.search(r'<determination>(.*?)</determination>', text, re.DOTALL)
actions = re.search(r'<action_items>(.*?)</action_items>', text, re.DOTALL)

print("\n" + "=" * 70)
print("PARSED SECTIONS")
print("=" * 70)

if determination:
    print(f"\n>>> DETERMINATION (what goes to the system):")
    print(determination.group(1).strip())

if reasoning:
    word_count = len(reasoning.group(1).split())
    print(f"\n>>> REASONING ({word_count} words — stored for audit trail)")

if actions:
    print(f"\n>>> ACTION ITEMS (what goes to the workflow queue):")
    print(actions.group(1).strip())

# --- Technique 2: JSON with embedded reasoning ---
print("\n" + "=" * 70)
print("TECHNIQUE 2: JSON with Embedded Reasoning")
print("=" * 70)

json_cot_prompt = f"""Analyze this auto liability claim. Return ONLY valid JSON,
no markdown fencing, with this structure:

{{
  "reasoning_steps": [
    {{"step": "string", "analysis": "string", "conclusion": "string"}}
  ],
  "determination": {{
    "liability_split": {{"our_insured_pct": number, "other_party_pct": number}},
    "rationale": "string (one sentence)",
    "confidence": "string (HIGH | MEDIUM | LOW)"
  }},
  "reserves": {{
    "property_damage": number,
    "bodily_injury": number,
    "total": number
  }},
  "action_items": ["string"],
  "risk_flags": ["string"]
}}

CLAIM:
{liability_scenario}"""

result_json = ask(json_cot_prompt)
print(result_json['text'])
print(f"\n[Tokens: {result_json['input_tokens']} in / {result_json['output_tokens']} out]")

try:
    parsed = parse_llm_json(result_json['text'])
    print("\n✓ Valid JSON")
    print(f"\n>>> LIABILITY SPLIT: Ours {parsed['determination']['liability_split']['our_insured_pct']}% / Theirs {parsed['determination']['liability_split']['other_party_pct']}%")
    print(f">>> CONFIDENCE: {parsed['determination']['confidence']}")
    print(f">>> RESERVES: ${parsed['reserves']['total']:,.2f}")
    print(f">>> REASONING STEPS: {len(parsed['reasoning_steps'])}")
    for step in parsed['reasoning_steps']:
        print(f"    → {step['step']}: {step['conclusion']}")
except json.JSONDecodeError as e:
    print(f"\n✗ JSON parse error: {e}")

# --- Comparison ---
print("\n" + "=" * 70)
print("XML vs JSON FOR CHAIN-OF-THOUGHT")
print("=" * 70)
print("""
XML TAGS are great when:
  • Output is mostly free-text that humans will read
  • You want flexible section lengths
  • Reasoning needs to be narrative/detailed
  → Think: adjuster notes, audit documentation

JSON is great when:
  • Downstream systems will consume the output
  • You need typed fields (numbers, booleans, arrays)
  • Reasoning steps need to be countable/iterable
  → Think: claims queue, dashboard, API response
""")

In [ ]:
# =============================================================================
# CoT for Multi-Step Insurance Calculations
# =============================================================================
# Where CoT really shines: math-heavy tasks where skipping a step means
# getting the wrong number. Like asking someone to show their work on a
# reserve calculation — the answer matters, but so does the math trail.
# =============================================================================

# --- Scenario: Complex reserve calculation ---
reserve_scenario = """
RESERVE CALCULATION REQUEST

Claim: WC-2026-07721 (Workers' Compensation)
Claimant: David Kowalski, age 42, warehouse supervisor
Employer: National Distribution Corp.
Date of Injury: January 8, 2026

Injury: Herniated disc (L4-L5) sustained while lifting heavy crate. Surgery 
recommended by treating physician. Currently on temporary total disability.

Wage Information:
- Pre-injury average weekly wage: $1,380
- State compensation rate: 66.67% of AWW
- State maximum weekly benefit: $1,100
- State minimum weekly benefit: $275

Medical Information:
- Emergency room visit: $4,200
- MRI and diagnostics: $3,800
- Pain management (8 visits to date): $6,400
- Recommended surgery (lumbar microdiscectomy): $45,000-$55,000
- Post-surgery physical therapy (estimated 16 weeks, 3x/week): $150/session
- Prescription medications to date: $1,850

Disability Timeline (estimated):
- Pre-surgery temporary total disability: 8 weeks (4 already elapsed)
- Post-surgery recovery/TTD: 12-16 weeks
- Potential permanent partial disability rating: 10-15% of whole person

State PPD Rate: $350/week, maximum 500 weeks at applicable percentage

Additional Factors:
- Claimant has 15 years with employer, good work history
- No prior workers' comp claims
- Employer wants to accommodate return-to-work with light duty
- IME (Independent Medical Exam) not yet scheduled
"""

# --- Without CoT: Just ask for the number ---
print("=" * 70)
print("APPROACH 1: Direct Calculation (No CoT)")
print("=" * 70)

direct_calc = f"""Calculate the total estimated reserve for this workers' 
compensation claim. Provide a single total number.

{reserve_scenario}"""

result_direct = ask(direct_calc)
print(result_direct['text'])
print(f"\n[Tokens: {result_direct['input_tokens']} in / {result_direct['output_tokens']} out]")

# --- With CoT: Step-by-step calculation ---
print("\n" + "=" * 70)
print("APPROACH 2: Step-by-Step Calculation (CoT)")
print("=" * 70)

cot_calc = f"""Calculate the total estimated reserve for this workers' compensation 
claim. Show every step of your math so it can be audited.

Work through these steps:
1. Calculate the weekly compensation rate (apply state max/min)
2. Calculate total temporary total disability (TTD) cost
3. Calculate all medical costs (past + projected)
4. Calculate permanent partial disability (PPD) exposure
5. Sum all components into a total reserve
6. Provide a low/mid/high range

For each step, show the formula, plug in the numbers, and state the result.

{reserve_scenario}"""

result_cot = ask(cot_calc)
print(result_cot['text'])
print(f"\n[Tokens: {result_cot['input_tokens']} in / {result_cot['output_tokens']} out]")

# --- Technique 3: CoT with JSON output for system consumption ---
print("\n" + "=" * 70)
print("APPROACH 3: CoT Reasoning → JSON Reserve Summary")
print("=" * 70)

hybrid_calc = f"""Calculate the total estimated reserve for this workers' compensation 
claim.

First, think through each component step-by-step in a <calculation> section.
Then provide the final reserve breakdown as JSON in a <reserve_json> section.

Use this JSON schema:
{{
  "weekly_comp_rate": number,
  "ttd": {{"weeks": number, "total": number}},
  "medical": {{
    "incurred_to_date": number,
    "projected_surgery": number,
    "projected_pt": number,
    "projected_meds": number,
    "medical_total": number
  }},
  "ppd": {{
    "weekly_rate": number,
    "weeks_low": number,
    "weeks_high": number,
    "ppd_low": number,
    "ppd_high": number
  }},
  "total_reserve": {{
    "low": number,
    "mid": number,
    "high": number
  }}
}}

{reserve_scenario}"""

result_hybrid = ask(hybrid_calc)
print(result_hybrid['text'])
print(f"\n[Tokens: {result_hybrid['input_tokens']} in / {result_hybrid['output_tokens']} out]")

# Parse the JSON from the hybrid response
json_match = re.search(r'<reserve_json>(.*?)</reserve_json>', result_hybrid['text'], re.DOTALL)
if json_match:
    try:
        reserve = parse_llm_json(json_match.group(1))
        print("\n" + "=" * 70)
        print("PARSED RESERVE SUMMARY")
        print("=" * 70)
        print(f"\n  Weekly Comp Rate:    ${reserve['weekly_comp_rate']:>10,.2f}")
        print(f"  TTD ({reserve['ttd']['weeks']} weeks):      ${reserve['ttd']['total']:>10,.2f}")
        print(f"  Medical Total:       ${reserve['medical']['medical_total']:>10,.2f}")
        print(f"  PPD Range:           ${reserve['ppd']['ppd_low']:>10,.2f} — ${reserve['ppd']['ppd_high']:,.2f}")
        print(f"  {'─' * 45}")
        print(f"  TOTAL RESERVE LOW:   ${reserve['total_reserve']['low']:>10,.2f}")
        print(f"  TOTAL RESERVE MID:   ${reserve['total_reserve']['mid']:>10,.2f}")
        print(f"  TOTAL RESERVE HIGH:  ${reserve['total_reserve']['high']:>10,.2f}")
    except json.JSONDecodeError as e:
        print(f"\n✗ JSON parse error: {e}")

# --- Why CoT matters for calculations ---
print("\n" + "=" * 70)
print("WHY CoT MATTERS FOR CALCULATIONS")
print("=" * 70)
print("""
Without CoT, the model may:
  • Skip steps and get wrong totals
  • Use wrong rates (forget to apply state max/min caps)
  • Mix up time periods
  • Miss cost components entirely

With CoT, you get:
  • Auditable math trail (regulators love this)
  • Catch errors at the step level, not just the total
  • Consistent methodology across claims
  • Low/mid/high ranges instead of false precision

The hybrid approach (CoT reasoning + JSON output) gives you BOTH:
  → Human-readable audit trail in <calculation>
  → Machine-readable numbers in <reserve_json>
""")